# Fine-Tuning TinyLlama-1.1B untuk Asisten K3 (Keselamatan & Kesehatan Kerja)

## 1. Instalasi Pustaka (Dependencies)

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install transformers datasets accelerate
# Pustaka untuk extract teks PDF dan export GGUF
!pip install pypdf llama-cpp-python

## 2. WAJIB: Upload File PDF Anda!

In [ ]:
import os

# Otomatis buat folder docs jika belum ada
os.makedirs("/content/docs", exist_ok=True)

pdf_files = [f for f in os.listdir("/content/docs") if f.endswith('.pdf')]
if not pdf_files:
    print("Hati-hati! Belum ada file PDF di folder /content/docs.")
    print("Silakan upload file PDF Anda terlebih dahulu.")
else:
    print(f"Ditemukan {len(pdf_files)} file PDF:")
    for p in pdf_files:
        print(f" - {p}")

## 3. Proses PDF Menjadi Dataset (Extract Q&A)

In [ ]:
import json
import glob
from pypdf import PdfReader

dataset_qa = [
    {"instruction": "Apa yang harus dilakukan saat mendengar alarm kebakaran?", "response": "Saat alarm kebakaran berbunyi:\n1. Tetap tenang dan jangan panik.\n2. Matikan semua peralatan listrik dan hentikan pekerjaan.\n3. Keluar dari ruangan melalui jalur evakuasi terdekat.\n4. JANGAN gunakan lift/elevator, gunakan tangga darurat.\n5. Berkumpullah di Assembly Point (Titik Kumpul) dan laporkan kehadiran Anda ke petugas Evakuasi (Floor Warden)."},
    {"instruction": "Bagaimana cara menggunakan APAR yang benar?", "response": "Gunakan teknik PASS saat memakai Alat Pemadam Api Ringan (APAR):\n- Pull: Tarik pin pengaman.\n- Aim: Arahkan ujung selang (nozzle) ke pangkal api.\n- Squeeze: Tekan tuas pengontrol (handle) perlahan.\n- Sweep: Sapukan dari sisi ke sisi permukaan api.\n\nJaga jarak aman sekitar 1,5 hingga 2 meter dari sumber api."},
    {"instruction": "Apa saja jenis-jenis api (kelas kebakaran) dan APAR yang cocok?", "response": "Kebakaran dibagi menjadi beberapa kelas:\n- Kelas A (Benda Padat: kayu, kertas, plastik): Gunakan APAR Air, Busa, atau Serbuk Kimia Kering (Dry Powder).\n- Kelas B (Zat Cair/Gas mudah terbakar: bensin, cat, LPG): Gunakan APAR Busa, CO2, atau Dry Powder.\n- Kelas C (Kelistrikan/Elektronik): Gunakan APAR CO2 atau Dry Powder. JANGAN GUNAKAN AIR!\n- Kelas D (Logam: magnesium, titanium): Gunakan APAR Khusus Logam (Dry Powder khusus)."},
    {"instruction": "Apa fungsi Assembly Point atau Titik Kumpul?", "response": "Assembly Point atau Titik Kumpul adalah area aman di luar gedung yang telah ditentukan sebagai zona evakuasi akhir. Fungsinya adalah sebagai lokasi berkumpul sementara agar petugas K3 atau Floor Warden dapat mendata dan memastikan tidak ada orang yang tertinggal di dalam gedung yang terbakar."},
    {"instruction": "Bagaimana cara mengecek kondisi APAR?", "response": "Cara inspeksi APAR meliputi:\n1. Periksa jarum indikator tekanan (pressure gauge), pastikan berada di area HIJAU.\n2. Pastikan pin pengaman dan segel masih terpasang dengan baik.\n3. Cek kondisi tabung luar, pastikan tidak ada karat atau kebocoran.\n4. Cek selang/nozzle, pastikan tidak ada retak atau sumbatan.\n5. Perhatikan label masa kedaluwarsa (expired date). Inspeksi ini harus dilakukan minimal 1 bulan sekali."},
    {"instruction": "Di mana lokasi yang tepat untuk meletakkan APAR?", "response": "APAR harus diletakkan di:\n1. Dekat dengan jalur evakuasi atau pintu keluar (exit).\n2. Terlihat jelas, mudah diakses, dan tidak terhalang barang apapun.\n3. Digantung pada dinding dengan jarak ketinggian ujung atas APAR sekitar 1,2 meter dari lantai.\n4. Dilengkapi dengan tanda/rambu segitiga merah APAR di atasnya."},
    {"instruction": "Apa itu sistem IoT dan mengapa memakai sensor gas MQ2 dan MQ7 untuk deteksi kebakaran?", "response": "Sistem IoT (Internet of Things) deteksi kebakaran biasanya menggunakan sensor MQ2 untuk mendeteksi keberadaan asap dan gas mudah terbakar seperti metana. Sedangkan MQ7 khusus dan sangat sensitif untuk mendeteksi Gas Karbon Monoksida (CO). Pembakaran benda organik saat kebakaran menghasilkan asap tebal dan CO dalam jumlah sangat tinggi. Kombinasi sensor ini mencegah false-alarm sekaligus mendeteksi gejala awal api lebih cepat."},
    {"instruction": "Apa saja ciri-ciri kondisi Waspada dan Kondisi Bahaya pada dashboard sistem?", "response": "- Kondisi Aman: Seluruh rasio sensor masih di ambang normal dan Computer Vision (YOLO) tidak menemukan objek api/asap.\n- Kondisi Waspada: Beberapa sensor MQ mengalami kenaikan di atas rata-rata udara bersih, atau YOLO mendeteksi sekumpulan asap namun belum ada titik api pasti (probabilitas > 50%).\n- Kondisi Bahaya: Sensor CO (MQ7) menembus ambang batas ekstrem disertai deteksi pasti api oleh YOLO (Probabilitas > 70%). Sistem akan membunyikan alarm otomatis."},
    {"instruction": "Apa yang harus saya lakukan sebagai petugas kontrol jika CCTV YOLO mendeteksi api?", "response": "Jika AI (YOLO) mendeteksi api:\n1. Segera validasi deteksi dengan melihat live-feed/streaming dari CCTV tersebut di dashboard.\n2. Jika api terlihat nyata, segera klik tombol 'Kirim Notifikasi Telegram' atau bunyikan alarm evakuasi secara paksa (Override) dari dashboard.\n3. Hubungi petugas tanggap darurat/satpam terdekat dengan kamera tersebut melalui radio komunikasi untuk segera menuju lokasi membawa APAR."},
    {"instruction": "Bagaimana cara melakukan pertolongan pertama pada korban yang menghirup asap kebakaran?", "response": "Pertolongan pertama untuk korban intoksikasi (keracunan) asap:\n1. Pindahkan korban segera ke area luar bangunan yang memiliki udara segar tertutup (up-wind).\n2. Longgarkan pakaian di sekitar area dada dan leher korban untuk melancarkan pernapasan.\n3. Jika korban sadar, instruksikan mengambil napas dalam secara perlahan.\n4. Jika korban tidak sadar, segera periksa nadi dan pernapasan. Jika tidak bernapas, lakukan CPR (Resusitasi Jantung Paru) jika Anda sudah terlatih.\n5. Segera hubungi tim medis darurat (Ambulans 118 atau 112)."},
    {"instruction": "Jelaskan Pertolongan Pertama Pada Kecelakaan (P3K) untuk korban luka bakar ringan!", "response": "Untuk luka bakar tingkat 1 dan 2 ringan:\n1. Jauhkan korban dari sumber api/panas.\n2. Aliri luka bakar dengan air mengalir (bukan es atau air es) selama 15-20 menit untuk memutus sisa hantaran panas di jaringan kulit.\n3. Lepaskan perhiasan atau pakaian di sekitar luka SEBELUM area luka membengkak (jangan paksa melepaskan pakian yang lengket ke kulit).\n4. Tutup luka menggunakan kasa steril ringan (jangan ditekan).\n5. Jangan oleskan odol, mentega, atau kopi pada luka bakar karena menyebabkan risiko infeksi berat!"},
    {"instruction": "Sebutkan nama-nama anggota tim developer proyek deteksi kebakaran ini?", "response": "Proyek Sistem Deteksi Kebakaran (PBL Semester 6) ini dikembangkan dengan dedikasi penuh oleh tim yang hebat: Ervin, Akmal, Jascon, dan Farhan."},
    {"instruction": "Apa fungsi dari sensor MQ135 dalam sistem?", "response": "Sensor MQ135 digunakan dalam sistem ini utamanya untuk mengevaluasi Kualitas Udara secara menyeluruh (Air Quality). Sensor ini sensitif terhadap berbagai gas berbahaya ringan, uap alkohol, hingga polusi asap ringan, memberikan peringatan awal sebelum api membesar."},
    {"instruction": "Apakah sistem bisa berjalan tanpa Ollama?", "response": "Ya, saya didesain untuk berjalan sepenuhnya LOKAL tanpa gateway LLM eksternal seperti Ollama. Model saya (TinyLlama GGUF) dimuat langsung di memori oleh Python באמצעות modul llama-cpp-python, sehingga menjamin kerahasiaan dan kecepatan yang maksimal dengan resource yang hemat."}
]

def chunk_text(text, max_len=400):
    sentences = text.split('.')
    chunks = []
    cur = ""
    for s in sentences:
        s = s.strip()
        if not s: continue
        if len(cur) + len(s) < max_len:
            cur += s + ". "
        else:
            if cur: chunks.append(cur.strip())
            cur = s + ". "
    if cur: chunks.append(cur.strip())
    return chunks

pdf_files = glob.glob("/content/docs/*.pdf")
for pdf_file in pdf_files:
    filename = os.path.basename(pdf_file)
    try:
        reader = PdfReader(pdf_file)
        full_text = ""
        for page in reader.pages:
            txt = page.extract_text()
            if txt: full_text += txt + " "
        import re
        full_text = re.sub(' +', ' ', full_text.replace("\n", " ").replace("\r", " "))
        
        chunks = chunk_text(full_text, 400)
        for i, chunk in enumerate(chunks):
            if len(chunk) < 50: continue
            q = f"Bagaimana panduan K3 terkait dokumen {filename}?"
            if i % 3 == 0: q = f"Berdasarkan regulasi di {filename}, apa yang perlu diketahui?"
            elif i % 2 == 0: q = f"Sebutkan instruksi keselamatan spesifik dari materi K3 berikut."
            dataset_qa.append({"instruction": q, "response": chunk})
    except Exception as e:
        print(f"Error reading {filename}: {e}")

with open("dataset.jsonl", "w", encoding="utf-8") as f:
    for d in dataset_qa:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

print(f"\nSUCCESS! dataset.jsonl dibuat dengan {len(dataset_qa)} pasang Q&A.")

## 4. Load Model TinyLlama & Setup Unsloth LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/tinyllama-chat",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
)

## 5. Mapping Prompts (ChatML)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

prompt = """<|system|>
Anda adalah Asisten K3 Pintar, profesional K3 (Keselamatan dan Kesehatan Kerja) dan Tanggap Darurat. Memberikan instruksi singkat, tegas, dan akurat.</s>
<|user|>
{}</s>
<|assistant|>
{}</s>"""

EOS_TOKEN = tokenizer.eos_token 
def formatting_prompts_func(examples):
    texts = []
    for inst, resp in zip(examples["instruction"], examples["response"]):
        texts.append(prompt.format(inst, resp))
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched=True)
print("Mapping Prompt Berhasil!")
print("Contoh:\n", dataset["text"][0])

## 6. Mulai Fine-Tuning (SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, 
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3, # Jumlah iterasi baca total data (tambahkan jika data sangat banyak)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 7. Export GGUF
Menyimpan ke dalam format paling ringan dan native bagi server Python kita.

In [ ]:
model.save_pretrained("lora_model") 
tokenizer.save_pretrained("lora_model")

print("Memulai Export ke GGUF (Estimasi: 3-5 menit)...")
model.save_pretrained_gguf("/content/K3_Chatbot_GGUF_Output", tokenizer, quantization_method="q4_k_m")

import os
print("\n\n====== SELESAI ======")
print("Silakan cek menu folder sidebar kiri di Colab, lalu masuk ke /content/K3_Chatbot_GGUF_Output/")
print("Download filenya, beri nama tinyllama-1.1b-chat.Q4_K_M.gguf lalu letakkan di dalam folder chatbot_model/ project Anda.")